In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ASTEncoder(nn.Module):
    def __init__(self, d_model, n_heads, rel_pos_emb_dim):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.rel_pos_emb = nn.Embedding(512, rel_pos_emb_dim)  # max distance 512
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.fc = nn.Linear(d_model + rel_pos_emb_dim, d_model)

    def forward(self, F_ast, sibling_matrix, ancestor_matrix, distance_matrix):
        """
        F_ast: [batch, n_nodes, d_model] - node features from preorder traversal
        sibling_matrix: [batch, n_nodes, n_nodes] - binary mask for siblings
        ancestor_matrix: [batch, n_nodes, n_nodes] - binary mask for ancestor-descendant
        distance_matrix: [batch, n_nodes, n_nodes] - int distances for rel pos emb
        """
        # Get relative positional embeddings
        rel_pos_emb = self.rel_pos_emb(distance_matrix)  # [batch, n_nodes, n_nodes, rel_pos_emb_dim]

        # Aggregate strong relationships (siblings + ancestor-descendant)
        strong_rel_mask = (sibling_matrix | ancestor_matrix).bool()  # [batch, n_nodes, n_nodes]

        # Decoupled attention: mask attention to only strong relationships
        attn_mask = ~strong_rel_mask  # invert for attn mask (True = block)
        # attn_mask should be [batch, n_nodes, n_nodes] for batch_first=True

        # Multihead attention (self-attention)
        attn_output, _ = self.attn(F_ast, F_ast, F_ast, attn_mask=attn_mask)

        # Concatenate with relative positional embeddings (mean over neighbors)
        rel_pos_emb_mean = (rel_pos_emb * strong_rel_mask.unsqueeze(-1)).sum(2) / (strong_rel_mask.sum(2, keepdim=True) + 1e-6)
        out = torch.cat([attn_output, rel_pos_emb_mean], dim=-1)
        out = self.fc(out)
        return out

In [3]:
# pip install transformers==4.* accelerate datasets torch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from transformers.modeling_outputs import BaseModelOutput

MODEL_NAME = "Salesforce/codet5p-220m"  # pick your codet5+ size

# ---------------------------
# 1) Wrapper model
# ---------------------------
class CodeT5pFromEmbeddings(nn.Module):
    def __init__(self, model_name=MODEL_NAME, proj_in_dim=None):
        super().__init__()
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        d_model = self.model.config.d_model
        # Optional projection if your embedding dim != d_model
        if proj_in_dim is not None and proj_in_dim != d_model:
            self.proj = nn.Linear(proj_in_dim, d_model)
        else:
            self.proj = None

    def forward(self, embeddings, attention_mask, labels=None):
        """
        embeddings: FloatTensor [B, S, H_in]
        attention_mask: LongTensor [B, S] (1=keep, 0=pad)
        labels: LongTensor [B, T] (token ids)
        """
        # Clean NaNs/Infs to keep training stable
        x = torch.nan_to_num(embeddings, nan=0.0, posinf=1e4, neginf=-1e4)

        if self.proj is not None:
            x = self.proj(x)  # -> [B, S, d_model]

        encoder_outputs = BaseModelOutput(last_hidden_state=x)
        out = self.model(
            encoder_outputs=encoder_outputs,
            encoder_attention_mask=attention_mask,
            labels=labels,                  # Trainer will compute cross-entropy
        )
        return out

    @torch.no_grad()
    def generate_from_embeddings(self, embeddings, attention_mask, **gen_kwargs):
        x = torch.nan_to_num(embeddings, nan=0.0, posinf=1e4, neginf=-1e4)
        if self.proj is not None:
            x = self.proj(x)
        encoder_outputs = BaseModelOutput(last_hidden_state=x)
        return self.model.generate(
            encoder_outputs=encoder_outputs,
            encoder_attention_mask=attention_mask,
            **gen_kwargs
        )

# ---------------------------
# 2) Toy dataset (replace with yours)
# ---------------------------
class EmbeddingSeq2SeqDataset(Dataset):
    """
    Each item:
      - 'emb': FloatTensor [S_i, H_in]  (your vectors for one example)
      - 'tgt': str                      (target sequence)
    """
    def __init__(self, items, tokenizer, max_target_len=256):
        self.items = items
        self.tok = tokenizer
        self.max_tgt = max_target_len

    def __len__(self): return len(self.items)

    def __getitem__(self, i):
        emb = self.items[i]["emb"]            # torch.FloatTensor [S_i, H_in]
        tgt = self.items[i]["tgt"]            # string (code/text)
        # tokenize target now (labels)
        y = self.tok(
            tgt,
            max_length=self.max_tgt,
            padding=False,
            truncation=True,
            return_tensors="pt"
        )
        labels = y.input_ids.squeeze(0)       # [T]
        return {"emb": emb, "labels": labels}

# ---------------------------
# 3) Collator: pad embeddings & labels to batch
# ---------------------------
class CollateEmbeddings:
    def __init__(self, tokenizer):
        self.tok = tokenizer

    def __call__(self, batch):
        # pad embeddings to longest S in batch
        embs = [b["emb"] for b in batch]  # list of [S_i, H]
        lengths = [e.size(0) for e in embs]
        H = embs[0].size(-1)
        padded_embs = pad_sequence(embs, batch_first=True)  # [B, S_max, H]
        # attention mask: 1 for real tokens, 0 for pad
        attn_mask = torch.zeros((len(embs), padded_embs.size(1)), dtype=torch.long)
        for i, L in enumerate(lengths):
            attn_mask[i, :L] = 1

        # pad labels (Trainer expects -100 for ignore index)
        labels_list = [b["labels"] for b in batch]  # each [T_i]
        labels_padded = pad_sequence(labels_list, batch_first=True,
                                     padding_value=self.tok.pad_token_id)
        labels_padded[labels_padded == self.tok.pad_token_id] = -100

        return {
            "embeddings": padded_embs,            # key must match model.forward
            "attention_mask": attn_mask,
            "labels": labels_padded,
        }

# ---------------------------
# 4) Instantiate tokenizer/model
# ---------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Ensure pad token exists for labels (CodeT5+ usually has '<pad>')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Your embedding dim (H_in). Replace with the real size.
H_IN = 1024  # e.g., your external encoder's hidden size
model = CodeT5pFromEmbeddings(MODEL_NAME, proj_in_dim=H_IN)

# ---------------------------
# 5) Build datasets
# ---------------------------
# EXAMPLE: make up two samples (replace with your real data)
S1 = 128  # sequence length of embeddings for sample 1
S2 = 80   # sequence length for sample 2
train_items = [
    {"emb": torch.randn(S1, H_IN), "tgt": "def add(a, b):\n    return a + b"},
    {"emb": torch.randn(S2, H_IN), "tgt": "print('hello world')"},
]
train_ds = EmbeddingSeq2SeqDataset(train_items, tokenizer)
collator = CollateEmbeddings(tokenizer)

# ---------------------------
# 6) Trainer
# ---------------------------
args = TrainingArguments(
    output_dir="./codet5p_from_vecs",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collator,
)

trainer.train()

# ---------------------------
# 7) Inference / generation
# ---------------------------
model.eval()
with torch.no_grad():
    test_emb = torch.randn(100, H_IN).unsqueeze(0)  # [1, S, H_in]
    test_mask = torch.ones(1, 100, dtype=torch.long)
    gen_ids = model.generate_from_embeddings(
        embeddings=test_emb,
        attention_mask=test_mask,
        max_new_tokens=64,
    )
    print(tokenizer.decode(gen_ids[0], skip_special_tokens=True))


KeyError: 'emb'

In [ ]:
!pip install 'accelerate>=0.26.0'

  Using cached accelerate-1.10.1-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.10.1-py3-none-any.whl (374 kB)
